# 03 - CTC local et controle qualite

Ce notebook ne rappelle jamais Mistral et ne demande aucune cle API. Il utilise les 34 transcriptions Mistral deja reussies, puis lance un modele CTC local compatible Windows/RTX 5070 pour comparer les textes.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(PROJECT_ROOT)

## Verification du projet

In [ ]:
from audit_ctc_readiness import build_audit

audit = build_audit()
audit['counts'], audit['device'], audit['tools'], audit['optional_packages']

## Modele CTC local

Sur Windows, on utilise le modele Transformers Darija local plutot que OmniASR.

In [ ]:
from ctc_transcriber import detect_device, discover_successful_samples

CTC_MODEL = 'boumehdi/wav2vec2-large-xlsr-moroccan-darija'
print('modele CTC retenu:', CTC_MODEL)
print('device:', detect_device())
print('samples disponibles:', len(discover_successful_samples()))

## Lancement CTC local

Cette cellule ne consomme pas l'API Mistral. `LIMIT = 0` traite tous les samples disponibles. `batch_size=1` reste prudent pour la VRAM de la RTX 5070 Laptop.

In [ ]:
from ctc_transcriber import run_ctc_pipeline

# LIMIT = 0 veut dire : traiter tous les audios Mistral reussis.
LIMIT = 0

ctc_results = run_ctc_pipeline(
    model=CTC_MODEL,
    limit=LIMIT,
    batch_size=1,
)
ctc_results

## Annotation et calibration

Apres le CTC, completez manuellement `outputs/ctc_human_annotations.csv`. Les valeurs autorisees sont `good`, `medium` et `reject`. Ne lancez la calibration qu'avec au moins 30 annotations et quatre videos distinctes.